# Week 3 — JOINs: `LEFT JOIN` and NULL Handling
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo (Thursday)**

By the end of this session, you will be able to:
- Write a `LEFT JOIN` that keeps **every** row of the left table, and read the `NULL`s
  it produces where the right table had no match
- Use the **anti-join** pattern — `LEFT JOIN ... WHERE right.key IS NULL` — to find the
  rows an `INNER JOIN` would have silently thrown away
- Handle `NULL` correctly across a join: `IS NULL` (never `= NULL`), and why a filter on
  the right table belongs in `ON`, not `WHERE`
- Combine **three tables** in a single query — `orders`, `customers` and `order_payments` —
  without letting the join fan out your counts

Yesterday you learned `INNER JOIN`, which keeps only what matches. Today is about
everything that *doesn't* match — which, in a real business, is usually where the
interesting problems are hiding.

### Run this first

The setup cell below loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same cell you ran yesterday — run it once, wait for
`Database ready.`, and leave it alone.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

Yesterday we ended on an uncomfortable discovery. Every revenue query you have written
so far joins `orders` to `order_items` — and an `INNER JOIN` keeps only the orders that
found a match. In the Olist dataset **775 orders have no line items at all**, so every
one of those queries has been quietly answering a slightly different question than the
one that was asked, with no error and no warning to tell you.

That is the shape of the most expensive mistake in analytics: not a query that crashes,
but a query that returns a confident, plausible, wrong number. An `INNER JOIN` is a
filter wearing a disguise.

A `LEFT JOIN` takes the disguise off. It keeps every row of the left table whether or
not the right table had anything to say about it, filling the right-hand columns with
`NULL` where there was no match. That one change turns a join from something that hides
gaps into something that **finds** them — which is why a `LEFT JOIN` is the first tool a
good analyst reaches for when auditing a new dataset.

## 1. `LEFT JOIN` — keep every row on the left

The syntax is one extra word. `FROM orders o LEFT JOIN order_items oi ON ...` reads
exactly like yesterday's join, and for every order that *does* have items you get the
same combined rows you got before. The difference is only visible at the edges: when an
order has no matching row in `order_items`, an `INNER JOIN` drops that order entirely,
while a `LEFT JOIN` keeps it and sets every `order_items` column to `NULL`.

The words "left" and "right" are literal — they mean the table written *before* the join
keyword and the table written *after* it. `orders LEFT JOIN order_items` protects
`orders`; flip the two table names around and you would be protecting `order_items`
instead. Deciding which table you are unwilling to lose rows from is the entire decision
behind choosing a `LEFT JOIN`.

The query below looks at canceled and unavailable orders, where both cases sit side by
side. Read the `order_item_id` and `price` columns: some rows carry real values, and
some come back empty. Those empty cells are the `NULL`s the `LEFT JOIN` invented, and
they are the rows yesterday's `INNER JOIN` was deleting.

In [ ]:
%%sql
-- Matched and unmatched rows side by side. Rows where order_item_id and price come
-- back empty are orders that have NO row at all in order_items.
SELECT o.order_id, o.order_status, oi.order_item_id, oi.price
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status IN ('canceled', 'unavailable')
ORDER BY o.order_id
LIMIT 8

## 2. How many rows does a `LEFT JOIN` return?

Yesterday's rule still applies — always ask what one row of the result represents — but
a `LEFT JOIN` adds a second term to the arithmetic. It returns **one row per matching
pair, plus one row for every left row that matched nothing**. So joining `orders` to
`order_items` gives you the 112,650 matched pairs the inner join produced, plus the 775
order-less orders it dropped: 113,425 rows in total.

The query below makes the two numbers visible in one result, and it does so with a
distinction worth committing to memory. `COUNT(*)` counts **rows**. `COUNT(oi.order_id)`
counts **non-NULL values of that column** — aggregate functions skip `NULL`s silently.
On a `LEFT JOIN` that gap between the two counts is precisely the number of unmatched
rows, which makes it a one-line health check on any join you write.

The third column is the sanity check from yesterday: 99,441 distinct orders means the
`LEFT JOIN` lost nothing at all.

In [ ]:
%%sql
-- Three ways to count the same result set. The gaps between them tell the story.
SELECT COUNT(*)                   AS rows_returned,    -- Expected: 113,425 = 112,650 + 775
       COUNT(oi.order_id)         AS rows_with_item,   -- Expected: 112,650 (NULLs skipped)
       COUNT(DISTINCT o.order_id) AS distinct_orders   -- Expected: 99,441 — every order kept
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id

## 3. The anti-join — isolating what didn't match

Now the pattern that earns `LEFT JOIN` its place in your toolkit. Keep every left row,
then filter down to **only** the rows where the right-hand key came back `NULL`. Because
a `NULL` on the right side can only mean "this row found no match", those are by
definition exactly the rows an `INNER JOIN` would have discarded. Analysts call this an
**anti-join**: a join used to find absence rather than presence.

Two details make or break it. First, filter on the **right** table's key
(`oi.order_id`), not the left one — `o.order_id` is never `NULL`, so filtering on it
would return nothing. Second, write `IS NULL`, never `= NULL`. `NULL` means "unknown",
and comparing anything to an unknown produces unknown rather than true, so `= NULL`
matches zero rows and reports a cheerful, silent `0`.

The three-line recipe is worth memorising, because you will write it for the rest of
your career: `LEFT JOIN` the table you suspect is incomplete, then
`WHERE that_table.key IS NULL`, then `COUNT`.

In [ ]:
%%sql
-- ANTI-JOIN: keep all orders, then keep only those order_items had no match for.
SELECT COUNT(*) AS orders_without_items   -- Expected: 775
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL                 -- IS NULL — never = NULL

## 4. Never stop at the count — go and look at the rows

775 is a finding, not an answer. The next question is always *what kind of rows are
these?*, and it is answered by swapping the `COUNT(*)` for a `GROUP BY` on the same
anti-join. The filter stays identical; only what you do with the surviving rows changes.

Run it and the mystery mostly dissolves: the overwhelming majority are `unavailable` and
`canceled` orders — orders that never reached fulfilment, so nothing was ever recorded
against them in `order_items`. That is not a bug in the data; it is the business showing
through the data, and it tells you these 775 rows can be safely excluded from a revenue
question.

Look at the tail of the list, though. A small number sit in statuses like `invoiced`,
`created` — and one is marked `shipped`. An order that shipped with no line item is a
genuine data-quality anomaly, the kind of thing worth raising with whoever owns the
source system. You would never have seen it with an `INNER JOIN`.

In [ ]:
%%sql
-- Same anti-join, GROUP BY instead of COUNT: what ARE these 775 orders?
SELECT o.order_status, COUNT(*) AS count
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL
GROUP BY o.order_status
ORDER BY count DESC

## 5. Three tables in one query

Nothing new to learn here — a third table is just another `JOIN ... ON` line. SQL joins
the first two tables into one wide intermediate result, then joins the third table onto
*that*, and you can keep chaining as long as each new table has a key linking it to
something already in the query.

The order of the lines follows the keys, not any rule about importance. Below, `orders`
links to `customers` through `customer_id`, and `orders` links to `order_payments`
through `order_id` — so `orders` sits in the middle as the hub both other tables hang
off. Sketching that little diagram on paper before writing a multi-table join saves a
lot of confusion later.

This one query answers something neither table could answer alone: *which payment
methods do customers in each state actually use?*

In [ ]:
%%sql
-- Three tables, two ON conditions: orders is the hub, customers and payments hang off it.
SELECT o.order_id, c.customer_state, op.payment_type, op.payment_value
FROM orders o
JOIN customers c      ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
LIMIT 10

## 6. Aggregating a three-table join without fanning out

Look carefully at the result above and you will notice the same `order_id` repeating
across consecutive rows with different `payment_type` values. That is not a mistake:
Olist lets a customer split one order across a credit card and two vouchers, so
`order_payments` holds **many rows per order**. The moment that table enters your query,
one order can occupy several rows.

This is yesterday's fan-out lesson arriving in a new costume, and it bites harder with
three tables because the duplication is one join further from where you are looking. The
defence is the same: never `COUNT(*)` after a one-to-many join when you mean to count
orders — write `COUNT(DISTINCT o.order_id)` so each order is counted once no matter how
many payment rows it produced.

One honest caveat about the result below: an order paid partly by voucher and partly by
card is counted once under *each* payment type it used, so the per-state numbers add up
to slightly more than that state's delivered-order count. That is the correct answer to
"how many delivered orders used this payment method", and it is worth saying out loud
when you present it — knowing what your number does and does not mean is the job.

In [ ]:
%%sql
-- Payment methods used by delivered orders in Brazil's three largest states.
-- COUNT(DISTINCT o.order_id) — NOT COUNT(*) — because order_payments fans out.
SELECT c.customer_state,
       op.payment_type,
       COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c       ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
  AND c.customer_state IN ('SP', 'RJ', 'MG')
GROUP BY c.customer_state, op.payment_type
ORDER BY c.customer_state, order_count DESC

## Going deeper — the anti-join as a data-quality audit

The anti-join is not a one-off trick for `order_items`. It is a general instrument: point
it at any child table and it answers *which parent rows are missing a record here?* Every
table that joins to `orders` on `order_id` can be audited with the same three lines, and
in a dataset you have just been handed, that audit is usually the most valuable half hour
you will spend.

Point it at `order_payments` and something odd falls out. `orders` has 99,441 rows and
`order_payments` covers 99,440 distinct orders, so **exactly one order in the entire
dataset has no payment record at all** — and its status is `delivered`. A delivered order
that was never paid for is not a rounding error you can shrug at; it is either a broken
record or a real hole in the money, and either way somebody needs to know.

Notice that the second column asks for the order's status without any aggregate. That is
legal here only because `COUNT(*)` returns a single row and there is a single matching
order behind it — with more than one match you would need a `GROUP BY`, which is why the
habit of running the count first and *then* looking at the rows is the safe order to work
in.

In [ ]:
%%sql
-- Audit a different child table with the identical pattern: which orders have no payment?
SELECT COUNT(*)     AS orders_without_payment,   -- Expected: 1  (99,441 - 99,440)
       o.order_id,
       o.order_status
FROM orders o
LEFT JOIN order_payments op ON o.order_id = op.order_id
WHERE op.order_id IS NULL

## Common mistakes

**Mistake 1 — `= NULL` instead of `IS NULL`.** This is the single most common `NULL` bug
in SQL, and it is nasty because it never errors. `NULL` is not a value, it is the absence
of one, so `something = NULL` evaluates to *unknown* — never true — and `WHERE` keeps only
rows that are definitively true. The result is a clean, confident `0` that looks like a
real finding. Any time a filter returns exactly zero rows and you expected some, check
your `NULL` comparison first. The same applies in reverse: use `IS NOT NULL`, never
`!= NULL`.

**Mistake 2 — filtering the right table in `WHERE`, which silently undoes your
`LEFT JOIN`.** This one is subtler and costs people whole afternoons. You write a
`LEFT JOIN` to keep all 99,441 orders, then add `WHERE oi.price > 100` — and your row
count collapses to 40,313. Why? Because unmatched rows have `oi.price = NULL`, and
`NULL > 100` is unknown, so `WHERE` throws every unmatched row away. Your `LEFT JOIN` has
been demoted to an `INNER JOIN` by a filter written three lines below it.

The fix is to move the condition into the `ON` clause. A condition in `ON` decides *what
counts as a match* while the join is being built, so unmatched left rows survive with
`NULL`s intact; a condition in `WHERE` runs *after* the join and deletes rows. The rule
to remember: **filters on the left table go in `WHERE`; filters on the right table of a
`LEFT JOIN` go in `ON`.**

In [ ]:
%%sql
-- ── COMMON MISTAKE 1: = NULL never matches anything ─────────────────
-- WRONG — returns 0 with no error, because NULL is never "equal" to anything:
--   SELECT COUNT(*) AS n FROM orders WHERE order_delivered_customer_date = NULL
-- ALSO WRONG, same reason:
--   SELECT COUNT(*) AS n FROM orders WHERE order_delivered_customer_date != NULL
-- CORRECT — IS NULL is the only way to test for absence:
SELECT COUNT(*) AS undelivered_orders
FROM orders
WHERE order_delivered_customer_date IS NULL   -- Expected: 2,965

In [ ]:
%%sql
-- ── COMMON MISTAKE 2: a WHERE on the right table kills the LEFT JOIN ─
-- WRONG — looks like a LEFT JOIN, behaves like an INNER JOIN. Unmatched rows
-- have oi.price = NULL, NULL > 100 is unknown, so WHERE deletes them. Returns
-- 40,313 rows instead of keeping all 99,441 orders:
--   SELECT COUNT(*) FROM orders o
--   LEFT JOIN order_items oi ON o.order_id = oi.order_id
--   WHERE oi.price > 100
-- CORRECT — put the right-table condition in ON, so every order survives:
SELECT COUNT(*)                   AS rows_returned,        -- Expected: 102,488
       COUNT(oi.order_id)         AS rows_with_pricey_item, -- Expected: 40,313
       COUNT(DISTINCT o.order_id) AS distinct_orders        -- Expected: 99,441 — nothing lost
FROM orders o
LEFT JOIN order_items oi
       ON o.order_id = oi.order_id
      AND oi.price > 100

## Mini-challenge — your turn

⏱ ~5–10 minutes

The customer experience team wants to run a follow-up campaign, and their first question
is a gap question: **how many orders never received a review at all?**

`order_reviews` is a child of `orders` on `order_id`, exactly like `order_items` was — so
this is the anti-join recipe from section 3 with one table name changed. Alias both
tables, `LEFT JOIN` from `orders`, filter on the **review** table's key being `NULL`, and
name your output column `orders_without_review`.

**Expected:** 768. You can predict it before you run it: `orders` has 99,441 rows and
`order_reviews` covers 98,673 distinct orders, so 99,441 − 98,673 = 768 orders have no
review. Getting the same number two different ways is how you know you got it right.

*Stretch, if you finish early:* change your query to a `GROUP BY o.order_status` — like
section 4 — and see whether review-less orders skew toward any particular status. Then
try replacing `IS NULL` with `= NULL` and watch your 768 turn into 0 with no complaint
from SQLite. That silence is the whole lesson.

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Group exercise

Open the Thursday exercises notebook and work through it in your groups. Five questions,
each with a self-check cell underneath — fill in the `%%sql` cell, run the check, and look
for the ✅.

They span everything from this week, not just today: a two-table join with a two-condition
filter (delivered orders from `MG` — you already know the number is **11,354**, so it is a
free check on your join), a three-table join for average payment value, a `GROUP BY` with
`HAVING` on `sellers`, and an average item price per review score.

Question 4 is deliberately open — *do orders with reviews have different average item
prices than orders without reviews?* — and it has no single right query. Before writing
anything, settle the argument in your group: which table goes on the left, and which join
type lets you see the "without reviews" group at all? An `INNER JOIN` cannot answer that
question, and understanding *why* is the point of the exercise.

## Session Summary

| Clause / idea | What it does | Example |
|---|---|---|
| `LEFT JOIN` | keeps every left row; unmatched right columns become `NULL` | `FROM orders o LEFT JOIN order_items oi ON o.order_id = oi.order_id` |
| left vs right | "left" = table before the keyword, "right" = table after it | `orders` is protected in the example above |
| anti-join | `LEFT JOIN` + `WHERE right.key IS NULL` = rows that matched nothing | `WHERE oi.order_id IS NULL` → 775 |
| `IS NULL` / `IS NOT NULL` | the only valid tests for absence (`= NULL` matches nothing) | `WHERE order_delivered_customer_date IS NULL` |
| `COUNT(*)` vs `COUNT(col)` | rows vs non-`NULL` values — the gap = unmatched rows | `COUNT(*)` 113,425 vs `COUNT(oi.order_id)` 112,650 |
| condition in `ON` | filters the match while joining; left rows survive | `ON o.order_id = oi.order_id AND oi.price > 100` |
| condition in `WHERE` | filters after joining; deletes unmatched rows | demotes a `LEFT JOIN` to an `INNER JOIN` |
| three-table join | chain another `JOIN ... ON`; use `COUNT(DISTINCT ...)` | `orders` → `customers` → `order_payments` |

**The two questions to ask before every join.** *Which table am I unwilling to lose rows
from?* — that answers `INNER` vs `LEFT`. And *does the right table have more than one row
per key?* — that answers whether `COUNT(*)` is lying to you.

---
**Coming up Wednesday (Week 4)**: `CASE WHEN` for building categories inside a query,
plus string and date functions — including `strftime` for pulling months and years out of
Olist's TEXT timestamps. Week 4 is also where **DeepSeek** is formally introduced, along
with the rule that governs it: attempt the query yourself first, then verify every
AI-drafted query against a known expected value before you trust it.